<a href="https://colab.research.google.com/github/FatherNurt/FPDP-Engine/blob/MAIN/137_Root_v2_RC1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 137 Root v2.0 RC1 — reproducibility audit
Hydrogen is the blueprint. Non-normative; run all. Requires numpy, scipy, mpmath. No external data. The provisional profile is not an empirical validation.

In [1]:
"""137 Root v2.0 RC1: standalone reference audit. Python 3.11+, numpy, scipy, mpmath."""
import math, json, platform
from pathlib import Path
import numpy as np
import scipy
from scipy.constants import alpha, physical_constants, elementary_charge, hbar
import mpmath as mp
mp.mp.dps=60
BASES={'phi':math.log((1+math.sqrt(5))/2),'2':math.log(2),'e':1.0}
MPBASES={'phi':mp.log((1+mp.sqrt(5))/2),'2':mp.log(2),'e':mp.mpf(1)}
Z=4*math.pi*alpha
K=1000; L=10; SEEDS=range(137,153)

def gate_float(y, noise, d, h):
    s=math.floor(y[0]/d+0.5); out=[s]
    for v,e in zip(y[1:],noise):
        z=v+e
        if z >= (s+0.5)*d+h: s+=1
        elif z <= (s-0.5)*d-h: s-=1
        out.append(s)
    return out

def gate_mp(y, noise, d, h):
    # independently expressed in address coordinates; shared decimal inputs
    h=mp.mpf(str(h))/d
    s=int(mp.floor(mp.mpf(str(y[0]))/d+mp.mpf('.5'))); out=[s]
    for v,e in zip(y[1:],noise):
        x=(mp.mpf(str(v))+mp.mpf(str(e)))/d
        up=x-s >= mp.mpf('.5')+h
        down=s-x >= mp.mpf('.5')+h
        s=s+int(up)-int(down); out.append(s)
    return out

def metrics(s,y,d,h,L=10):
    a=np.asarray(s); diff=np.diff(a); idx=np.flatnonzero(diff); n=len(diff); chi=len(idx)
    # Only departures with complete L subsequent samples eligible; L counts after departure.
    eligible=[int(k) for k in idx if k+1+L<=n]
    back=sum(a[k] in a[k+2:k+2+L] for k in eligible)
    residual=np.maximum(0, np.abs(np.asarray(y)-a*d)-(d/2+h))
    ideal=np.floor(np.asarray(y)/d+0.5).astype(int)
    N=int(ideal[-1]-ideal[0]); progress=int(a[-1]-a[0]);
    return dict(chi=chi, eligible=len(eligible), back=int(back), r_back=back/len(eligible) if eligible else None,
                N0=N, progress=progress, missed=max(0,N-progress), excess=max(0,chi-N),
                max_error=float(max(residual)), tracking_fraction=float(np.mean(residual[1:]<=0.02)),
                I=chi/N if N>0 else None)

def qualified(m,nu):
    # Provisional engineering profile, not physical law. No eligible returns => indeterminate unless zero toggles.
    back_ok=(m['chi']==0 or (m['r_back'] is not None and m['r_back']<=0.05))
    tracking=m['tracking_fraction']>=0.99 and m['max_error']<=0.10
    if nu==0: return tracking and back_ok and m['chi']<=2
    return tracking and back_ok and m['missed']<=1 and m['excess']<=max(1,0.10*m['N0'])

def counterexamples():
    y=[math.log(Z)+2*k for k in range(K+1)]
    fast={}
    for name,d in BASES.items():
        s=gate_float(y,[0]*K,d,.004); m=metrics(s,y,d,.004)
        assert m['chi']/(K*2/d)<=1.10 and not qualified(m,2)
        fast[name]=m
    s=[0,1,0,1]+[1]*12; y=[0]*len(s)
    m=metrics(s,y,1,0,L=10)
    old=sum(s[min(len(s)-1,k+10)]==s[k] for k in range(len(s)-1) if s[k+1]!=s[k])
    assert old==1 and m['back']==2
    assert metrics([0,0],[0,0],1,0)['r_back'] is None
    # Tie rules using exactly representable base-e thresholds.
    assert gate_float([0,.5,-.5],[0,0],1,0)==[0,1,0]
    assert gate_mp([0,.5,-.5],[0,0],mp.mpf(1),0)==[0,1,0]
    # Stateless classifier can be stable when bounded noise cannot reach threshold.
    assert all(math.floor(v+0.5)==0 for v in [-.1,0,.1])
    return {'fast_drift':fast,'return_fixture':{'old':old,'new':m['back']},'edge_tests':'PASS'}

def run():
    z_ind=physical_constants['characteristic impedance of vacuum'][0]*elementary_charge**2/hbar
    assert abs(z_ind/Z-1)<1e-8
    report={'versions':{'python':platform.python_version(),'numpy':np.__version__,'scipy':scipy.__version__,'mpmath':mp.__version__},'alpha_snapshot':alpha,'Ztilde':Z,'Z_from_impedance':z_ind,'relative_difference':abs(z_ind/Z-1),'counterexamples':counterexamples(),'grid':[]}
    for nu in [0,.001]:
      y=[math.log(Z)+k*nu for k in range(K+1)]
      for sig in [.001,.005,.02]:
       for h in [.002,.004,.008]:
        for name,d in BASES.items():
         rows=[]; agreement=True
         for seed in SEEDS:
          noise=np.random.default_rng(seed).normal(0,sig,K).tolist()
          s=gate_float(y,noise,d,h); t=gate_mp(y,noise,MPBASES[name],h)
          agreement &= s==t
          m=metrics(s,y,d,h); rows.append(m)
         count=sum(qualified(m,nu) for m in rows)
         report['grid'].append(dict(base=name,nu=nu,sigma=sig,h=h,runs=len(rows),agree=bool(agreement),qualified=count,profile_pass=bool(agreement and count==len(rows)),mean_chi=float(np.mean([m['chi'] for m in rows])),worst_tracking=min(m['tracking_fraction'] for m in rows)))
    assert all(r['agree'] for r in report['grid'])
    return report



In [ ]:
report=run()
Path('audit_results.json').write_text(json.dumps(report,indent=2,allow_nan=False))
print('Cases:',len(report['grid']),'runs:',sum(r['runs'] for r in report['grid']),'passes:',sum(r['profile_pass'] for r in report['grid']))
print('All state sequences agree:',all(r['agree'] for r in report['grid']))